In [6]:
# Sel 1 — import semua yang dibutuhkan
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import joblib
import os

In [8]:
# Sel 2 — muat dataset menu
df_menu = pd.read_excel('../data/menu_dataset.xlsx')

# Pastikan tidak ada nilai kosong di kolom numerik
kolom_numerik = ['kalori_kcal', 'protein_g', 'lemak_g', 'karbohidrat_g']
df_menu[kolom_numerik] = df_menu[kolom_numerik].fillna(df_menu[kolom_numerik].mean())

print(f"Total menu: {len(df_menu)}")
print(df_menu.head())

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [3]:
# Sel 3 — normalisasi data (wajib sebelum KNN)
# KNN pakai jarak — kalau satu kolom nilainya 0-500 dan lain 0-50,
# kolom pertama mendominasi perhitungan jarak. Scaler menyamakan skalanya.

scaler = StandardScaler()
X = df_menu[kolom_numerik].values
X_scaled = scaler.fit_transform(X)

print("Data sebelum scaling (5 baris pertama):")
print(X[:5])
print("\nData setelah scaling:")
print(X_scaled[:5])

Data sebelum scaling (5 baris pertama):
[[450  25  10  60]
 [350  18   8  45]
 [300  20   9  35]
 [420  28   7  58]
 [320  15   6  50]]

Data setelah scaling:
[[ 1.20604538  0.52133661  0.52758934  1.30654916]
 [-0.30151134 -0.49708839 -0.17586311 -0.20100756]
 [-1.05528971 -0.20610982  0.17586311 -1.20604538]
 [ 0.75377836  0.95780446 -0.52758934  1.1055416 ]
 [-0.75377836 -0.93355625 -0.87931557  0.30151134]]


In [4]:
# Sel 4 — latih model KNN
# n_neighbors=5: ambil 5 menu terdekat sebagai kandidat rekomendasi
model_knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
model_knn.fit(X_scaled)

print("Model KNN berhasil dilatih!")
print(f"Jumlah data latih: {model_knn.n_samples_fit_}")

Model KNN berhasil dilatih!
Jumlah data latih: 12


In [5]:
# Sel 5 — test rekomendasi manual sebelum disimpan
# Simulasi: cari menu mirip dengan target kalori 600 kcal, protein 25g
query = np.array([[600, 25, 15, 80]])  # kalori, protein, lemak, karbo
query_scaled = scaler.transform(query)

distances, indices = model_knn.kneighbors(query_scaled)

print("5 menu paling mirip dengan target nutrisi (kalori=600, protein=25g):")
print(df_menu.iloc[indices[0]][['nama_menu', 'jenis_diet', 'waktu_makan',
                                  'kalori_kcal', 'protein_g']])

5 menu paling mirip dengan target nutrisi (kalori=600, protein=25g):
                                nama_menu            jenis_diet  waktu_makan  \
9                      Nasi Tim Hati Ayam  Diet Tinggi Zat Besi  Makan Siang   
0         Nasi Merah + Ayam Kukus + Bayam               Diet DM  Makan Siang   
3  Nasi Putih + Sayur Bening + Ikan Kukus   Diet Rendah Natrium  Makan Siang   
5              Kentang Rebus + Ayam Rebus            Diet Lunak  Makan Siang   
6          Nasi + Telur Dadar Putih Telur     Diet Rendah Lemak  Makan Siang   

   kalori_kcal  protein_g  
9          480         28  
0          450         25  
3          420         28  
5          400         22  
6          380         24  


In [6]:
# Sel 6 — simpan model dan scaler ke folder models/
os.makedirs('../models', exist_ok=True)

joblib.dump(model_knn, '../models/knn_menu.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(df_menu, '../models/menu_data.pkl')

print("Model, scaler, dan data menu berhasil disimpan di folder models/")

Model, scaler, dan data menu berhasil disimpan di folder models/
